In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from scipy.stats import ttest_ind, chi2_contingency
import statsmodels.api as sm
from statsmodels.formula.api import ols

In [3]:
df = pd.read_csv("/content/cookie_cats.csv")

In [ ]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [7]:
# Check how many players are in each version
df["version"].value_counts()


,count
version,
gate_40,45489
gate_30,44700


# T-Test

We’ll use t-test to check if sum_gamerounds is different between the two versions.

In [5]:
#Separate data by version
'''df[df["version"]=="gate_30"] → filters all rows for gate_30
["sum_gamerounds"] → takes only the sum_gamerounds column from those filtered rows'''

gate_30 = df[df["version"]=="gate_30"]["sum_gamerounds"]
gate_40 = df[df["version"]=="gate_40"]["sum_gamerounds"]

In [6]:
# Perform t-test
'''
ttest_ind() → runs an independent t-test.
This test checks: Are the average rounds played different between gate_30 and gate_40

t_stat → tells the difference between the two averages in terms of standard deviations.
It compares the average rounds played in gate_30 vs gate_40
A large number → big difference
A small number → small difference

p_value → tells how likely this difference happened by chance.

If p_value < 0.05, the difference is real.
If p_value ≥ 0.05, the difference could be just random luck.
Small p_value (< 0.05) → difference is real
Big p_value (≥ 0.05) → difference is just luck
'''
t_stat, p_value = ttest_ind(gate_30, gate_40)
print("T-test results:")
print("T-statistic:", t_stat)
print("P-value:", p_value)

T-test results:
T-statistic: 0.8910426211362967
P-value: 0.37290868247405207


Since 0.3729 > 0.05, it means:

The difference in the number of rounds played between gate_30 and gate_40 is NOT statistically significant.

In simple words:

“Players in gate_40 didn’t really play more rounds than gate_30. The small difference we see could just happen by chance.

# Chi-Square Test

In [9]:
# Create contingency table
'''
pd.crosstab() creates a table of counts.
It counts how many players fall into each combination of:
version → gate_30 and gate_40
retention_1 → True/False (or 1/0)
'''
contingency_1day = pd.crosstab(df["version"], df["retention_1"])
print(contingency_1day)

retention_1  False  True 
version                  
gate_30      24666  20034
gate_40      25370  20119


In [13]:
# Chi-square test
'''
chi2_contingency is a statistical test function.
It checks whether two things are related or not.
(Does game version (gate_30 / gate_40) affect retention (yes / no)?)

chi2
→ Measures the difference between the two versions.
p
→ Tells whether the difference is real or by chance.
dof
→ degrees of freedom (technical, not needed to explain)
expected
→ expected counts if there was no difference


'''
chi2, p, dof, expected = chi2_contingency(contingency_1day)
print("\nChi-square test for 1-day retention:")
print("Chi-square:", chi2)
print("P-value:", p)


Chi-square test for 1-day retention:
Chi-square: 3.1591007878782262
P-value: 0.07550476210309086


0.075 > 0.05
This means the result is NOT statistically significant.

therefore


There is no significant relationship between game version (gate_30 and gate_40) and 1-day retention.

# Anova

In [14]:
# ANOVA example with sum_gamerounds
model = ols('sum_gamerounds ~ version', data=df).fit()
anova_table = sm.stats.anova_lm(model)
print("\nANOVA results:")
print(anova_table)



ANOVA results:
               df        sum_sq       mean_sq         F    PR(>F)
version       1.0  3.020603e+04  30206.031880  0.793957  0.372909
Residual  90187.0  3.431158e+09  38044.923945       NaN       NaN


ANOVA test shows F = 0.794 and p = 0.373, which is greater than 0.05. Therefore, we fail to reject the null hypothesis, meaning the two game versions do not have a significant difference in average gameplay rounds.